### Choose where this notebook runs

This is the **IQM edition**: circuit construction, drawings and analysis are identical to the IBM edition; the execution of Part 6 (the GHZ circuit) and the circuits of the In-depth analysis are transpiled to the IQM native gate set **PRX + CZ** and run on **Garnet** (20 qubits) or **Emerald** (54 qubits) via IQM Resonance. Pick the backend by editing the single `TARGET`
line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# Constructing Quantum Circuits

**Course**: Quantum Computing Laboratory  
**Version**: 2.2 (Qiskit 2.2 Compatible)  
**Last Updated**: November 2025

---

In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from iqm_backend import get_backend, run_counts` | the helper module of this folder: `run_counts(circuit, TARGET, shots=...)` = `transpile_to_IQM` (rewrite the circuit with the native `r` = PRX rotations and `cz` gates, routed on the chip's square lattice) → `backend.run` → counts as a plain dict; `run_counts(..., return_transpiled=True)` also returns the transpiled circuit |
| `TARGET = "fake_apollo"` | a local noisy simulator with the 20-qubit Garnet topology: no token, no credits, realistic gate and readout noise; `"garnet:mock"` checks the cloud pipeline but returns RANDOM bitstrings; `"garnet"` / `"emerald"` are the real QPUs (token + credits; IQM Resonance declines circuits above 10 000 native instructions) |
| `get_backend(TARGET)` | the raw Qiskit backend object (name, qubit count, coupling map) if you want to call `backend.run` yourself |

</details>


## 📚 Overview

Master the **QuantumCircuit** class - the foundation of all quantum programming in Qiskit. Learn to build, manipulate, and compose quantum circuits.

### Topics Covered:
- QuantumCircuit fundamentals
- Adding gates and operations
- Quantum and classical registers
- Circuit composition and decomposition
- Parameterized circuits
- Custom gates and instructions

## 🎯 Learning Objectives

1. ✅ Understand QuantumCircuit structure
2. ✅ Create circuits with multiple methods
3. ✅ Compose and decompose circuits
4. ✅ Build parameterized circuits
5. ✅ Create custom quantum gates
6. ✅ Apply circuits to real algorithms

In [ ]:
# Qiskit 2.2 imports
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister   # the circuit object and named registers of qubits / classical bits
from qiskit.circuit import Parameter, ParameterVector                  # symbolic angles (θ, φ) that get a value later
from qiskit.circuit.library import RZGate, CXGate, HGate               # gates as stand-alone objects (alternative to qc.rz / qc.cx / qc.h)
from qiskit.visualization import circuit_drawer                        # the function behind qc.draw()
# transpilation + execution are done by run_counts (imported above) on the IQM backend chosen by TARGET
import numpy as np                                                     # numerics: np.pi, arrays
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns

print("✓ All imports successful!")
print("  Qiskit 2.2 QuantumCircuit ready | TARGET =", TARGET)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumRegister`, `ClassicalRegister` | named groups of qubits / classical bits (Part 2); a plain `QuantumCircuit(n, m)` creates unnamed ones |
| `Parameter`, `ParameterVector` | symbols for gate angles: a circuit *template* whose numbers are filled in later (Part 4) |
| no simulator imports | in this edition Part 6 runs through `run_counts(circuit, TARGET, shots)` from `iqm_backend` (imported in the TARGET cell) |
| `draw(qc)` + `display(...)` | our helper around `qc.draw("mpl")`; `display()` shows a figure from the middle of a cell |

</details>


## 🔨 Part 1: Circuit Basics

### What is a QuantumCircuit?

In [ ]:
# Create a simple 2-qubit circuit
qc = QuantumCircuit(2)                              # QuantumCircuit(number_of_qubits): no classical bits, all qubits in |0⟩

print("Circuit created:")
print(f"  Qubits: {qc.qubits}")                     # list of Qubit objects (register + index)
print(f"  Number of qubits: {qc.num_qubits}")
print(f"  Classical bits: {qc.clbits}")             # empty list: nothing to store measurement results in yet
print(f"  Number of classical bits: {qc.num_clbits}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(2)` | an empty 2-qubit circuit; `QuantumCircuit(2, 2)` would add 2 classical bits for measurements |
| `qc.qubits` | the qubit objects: `Qubit(QuantumRegister(2, 'q'), 0)` = qubit 0 of the default register `q` |
| `f"...{expr}..."` | an f-string: Python evaluates `expr` and inserts it in the text |

</details>


### Adding Gates

In [ ]:
# Add gates to circuit
qc.h(0)  # Hadamard on qubit 0
qc.x(1)  # X-gate on qubit 1
qc.barrier()  # Visual separator

print("Circuit data:")
print(f"  Instructions: {len(qc.data)}")    # qc.data = the list of instructions (the barrier counts as one)
print(f"  Depth: {qc.depth()}")             # number of time steps: H and X act on different qubits, so they run in parallel
print(f"  Gate count: {qc.size()}")         # number of operations (barriers excluded)

# Visualize
display(draw(qc)); print("↑ H on q0 and X on q1 in the same time step, then a barrier (dashed line) across both qubits")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.h(0)`, `qc.x(1)` | methods that *append* a gate; the integer is the qubit index |
| `qc.barrier()` | a separator: no physical effect, but the transpiler does not move gates across it |
| `len(qc.data)` vs `qc.size()` | 3 instructions (barrier included) vs 2 gates (barrier excluded) |
| `qc.depth()` | the length of the critical path: 1, because the two gates touch different qubits |

</details>


### Circuit Instructions

In [ ]:
# Inspect circuit instructions
print("Circuit instructions:")
for i, instruction in enumerate(qc.data):          # enumerate gives (index, element) pairs
    print(f"  {i}: {instruction.operation.name} on qubits {instruction.qubits}")   # each item = operation + the qubits it acts on

# Get specific instruction
first_gate = qc.data[0]                            # list indexing: the first instruction
print(f"\nFirst gate: {first_gate.operation.name}")
print(f"  Applied to: {first_gate.qubits}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.data` | the circuit as a list of `CircuitInstruction`s, in the order they were appended |
| `instruction.operation.name` | the gate's name ('h', 'x', 'barrier'); `.operation` also holds its matrix and parameters |
| `instruction.qubits` | a tuple of Qubit objects: the barrier acts on both |

</details>


## 🧩 Part 2: Registers

### Using QuantumRegister and ClassicalRegister

In [ ]:
# Create named registers
qr1 = QuantumRegister(2, 'alice')                   # QuantumRegister(size, name)
qr2 = QuantumRegister(2, 'bob')
cr = ClassicalRegister(2, 'result')                 # 2 classical bits named 'result'

# Create circuit with registers
qc_reg = QuantumCircuit(qr1, qr2, cr)               # the circuit takes the registers in this order: alice_0, alice_1, bob_0, bob_1

print("Circuit with named registers:")
print(f"  Quantum registers: {qc_reg.qregs}")
print(f"  Classical registers: {qc_reg.cregs}")
print(f"  Total qubits: {qc_reg.num_qubits}")

# Add operations
qc_reg.h(qr1[0])                                    # a qubit can be addressed through its register
qc_reg.cx(qr1[0], qr2[0])                           # CNOT alice_0 -> bob_0: a Bell pair shared by Alice and Bob
qc_reg.measure(qr1, cr)                             # measure the whole alice register into the whole result register

display(draw(qc_reg)); print("↑ named wires alice/bob/result: a Bell pair between alice_0 and bob_0, then Alice's two qubits measured")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumRegister(2, 'alice')` | a named group of 2 qubits; names make larger circuits (teleportation, error correction) readable |
| `qr1[0]` | indexing a register: the first qubit of Alice |
| `qc_reg.measure(qr1, cr)` | register-to-register measurement: alice_i → result_i |

</details>


### Multiple Classical Registers

In [ ]:
# Multiple classical registers for different measurements
qr = QuantumRegister(3, 'q')
cr1 = ClassicalRegister(1, 'syndrome')
cr2 = ClassicalRegister(2, 'data')

qc_multi = QuantumCircuit(qr, cr1, cr2)

# Example: error correction syndrome measurement
qc_multi.h(0)                                      # integers still work: qubit 0 = q[0]
qc_multi.cx(0, 1)
qc_multi.cx(0, 2)                                  # (|000⟩ + |111⟩)/√2: a 3-qubit GHZ state
qc_multi.measure(qr[0], cr1[0])  # Syndrome
qc_multi.measure(qr[1:3], cr2)  # Data             (slicing a register: qubits 1 and 2)

print("Multiple classical registers:")
display(draw(qc_multi)); print("↑ two classical registers: q0 goes to 'syndrome', q1 and q2 to 'data'")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qr[1:3]` | slicing (end excluded): qubits 1 and 2 |
| two classical registers | the counts keys then contain one group of bits per register, separated by a space ('data syndrome', last register first) |
| `h` + two `cx` | the GHZ state: all three qubits agree, so the syndrome bit equals both data bits |

</details>


## 🔄 Part 3: Circuit Composition

### Building Modular Circuits

In [ ]:
# Create sub-circuits
bell_pair = QuantumCircuit(2, name='Bell')         # name= labels the block when it is drawn as a box
bell_pair.h(0)
bell_pair.cx(0, 1)

teleport = QuantumCircuit(3, name='Teleport')
teleport.cx(1, 2)
teleport.h(1)

# Compose circuits
full_circuit = QuantumCircuit(3)
full_circuit.compose(bell_pair, qubits=[1, 2], inplace=True)   # put Bell's qubits 0, 1 on our qubits 1, 2; inplace=True modifies full_circuit
full_circuit.barrier()
full_circuit.compose(teleport, inplace=True)                   # without qubits=: qubit i -> qubit i

print("Composed circuit:")
display(draw(full_circuit)); print("↑ compose() copies the gates of the sub-circuits (no boxes): Bell pair on q1, q2, then the Teleport gates")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `compose(other, qubits=[1, 2], inplace=True)` | append another circuit; `qubits` maps its qubits onto ours; without `inplace=True` a *new* circuit is returned |
| `name='Bell'` | only used when the circuit is turned into a gate (`to_gate()`, Part 5); compose() inserts the individual gates |

</details>


### Tensor Product of Circuits

In [ ]:
# Create two independent circuits
qc1 = QuantumCircuit(1)
qc1.h(0)

qc2 = QuantumCircuit(1)
qc2.x(0)

# Tensor product (parallel composition)
qc_tensor = qc1.tensor(qc2)                        # qc1 ⊗ qc2: qc2 goes on the LOW qubit (q0), qc1 on the high one (q1)

from qiskit.quantum_info import Statevector         # exact state of the circuit
print("Tensor product:")
print(f"  qc1 ⊗ qc2 creates {qc_tensor.num_qubits}-qubit circuit")
print(f"  state probabilities (keys = q1 q0): { {str(k): round(float(p), 3) for k, p in Statevector(qc_tensor).probabilities_dict().items()} }")   # Qiskit writes bitstrings with qubit 0 on the RIGHT
display(draw(qc_tensor)); print("↑ X (from qc2) on q0, H (from qc1) on q1: the right factor of ⊗ sits on the lowest qubit")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc1.tensor(qc2)` | the circuit for qc1 ⊗ qc2; Qiskit's convention puts the right-hand factor on qubit 0 |
| `Statevector(...).probabilities_dict()` | the exact outcome probabilities as {bitstring: p}, zero entries omitted; the dict comprehension `{str(k): round(float(p), 3) ...}` only tidies the printout |
| bitstring order | Qiskit is *little-endian*: in '01' the rightmost character is qubit 0 |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The tensor product, by hand.** $qc_2$ prepares $X|0\rangle = |1\rangle$ and $qc_1$ prepares $H|0\rangle = |+\rangle$. With $qc_1$ on qubit 1 and $qc_2$ on qubit 0 the state is

$$|{+}\rangle_{q_1}\otimes|1\rangle_{q_0} = \frac{1}{\sqrt2}\big(|0\rangle + |1\rangle\big)_{q_1}|1\rangle_{q_0} = \frac{1}{\sqrt2}\big(|01\rangle + |11\rangle\big),$$

written as $|q_1 q_0\rangle$. So the two outcomes '01' and '11' have probability $1/2$ each, and qubit 0 is always 1 — the printed dictionary `{'01': 0.5, '11': 0.5}`.

</details>


## 🎛️ Part 4: Parameterized Circuits

### Using Parameters

In [ ]:
# Create parameterized circuit
theta = Parameter('θ')                             # a symbol, not a number: the circuit becomes a template
phi = Parameter('φ')

qc_param = QuantumCircuit(2)
qc_param.rx(theta, 0)                              # RX by the (still unknown) angle θ
qc_param.ry(phi, 1)
qc_param.cx(0, 1)

print("Parameterized circuit:")
print(f"  Parameters: {qc_param.parameters}")      # the set of free symbols, sorted by name
print(f"  Number of parameters: {qc_param.num_parameters}")
display(draw(qc_param)); print("↑ the angles appear as symbols θ and φ: this circuit cannot run until they get values")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `Parameter('θ')` | a symbolic angle; the same circuit can then be evaluated for many values (variational algorithms do exactly this) |
| `qc_param.parameters` | a `ParameterView` of the free symbols; `num_parameters` = 2 |

</details>


### Binding Parameters

In [ ]:
# Bind specific values
qc_bound = qc_param.assign_parameters({theta: np.pi/4, phi: np.pi/2})   # dict {Parameter: value}; returns a NEW circuit

print("Circuit with bound parameters:")
print(f"  Remaining parameters: {qc_bound.parameters}")   # empty: every symbol has a number now
display(draw(qc_bound)); print("↑ the same circuit with θ = π/4 and φ = π/2 filled in")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `assign_parameters({theta: ..., phi: ...})` | substitute numbers for symbols; the original `qc_param` is unchanged (`inplace=False` by default) |
| empty `parameters` | a fully bound circuit can be simulated or executed |

</details>


### Parameter Vectors

In [ ]:
# Create parameter vector
params = ParameterVector('p', 4)                   # four parameters p[0] … p[3] under one name

qc_vec = QuantumCircuit(2)
qc_vec.rx(params[0], 0)
qc_vec.ry(params[1], 0)
qc_vec.rx(params[2], 1)
qc_vec.ry(params[3], 1)
qc_vec.cx(0, 1)

print("Circuit with parameter vector:")
print(f"  Parameters: {qc_vec.parameters}")
display(draw(qc_vec)); print("↑ four angles p[0] … p[3]; a ParameterVector can be bound at once with a list of 4 numbers")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `ParameterVector('p', 4)` | a list-like group of parameters: convenient for ansätze with many angles |
| `params[0]` | indexing gives the single `Parameter` p[0] |
| binding a vector | `qc_vec.assign_parameters([0.1, 0.2, 0.3, 0.4])` binds them in order |

</details>


## 🏗️ Part 5: Custom Gates

### Creating Custom Gates

In [ ]:
# Define custom gate as a circuit
def create_custom_gate():
    qc = QuantumCircuit(2, name='MyGate')          # the name shown on the box
    qc.h(0)
    qc.cx(0, 1)
    qc.rz(np.pi/4, 1)
    qc.cx(0, 1)
    return qc.to_gate()                            # freeze the circuit into a reusable Gate object

# Create the gate
my_gate = create_custom_gate()

# Use custom gate
qc_custom = QuantumCircuit(3)
qc_custom.h(0)
qc_custom.append(my_gate, [0, 1])                  # append(gate, qubits): the gate's qubit 0 -> our qubit 0, its qubit 1 -> our 1
qc_custom.append(my_gate, [1, 2])                  # the same gate again on qubits 1, 2

print("Circuit with custom gate:")
display(draw(qc_custom)); print("↑ MyGate appears as one box, applied twice on overlapping qubit pairs")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.to_gate()` | turns a (measurement-free) circuit into a `Gate` that can be appended like `h` or `cx` |
| `qc_custom.append(my_gate, [0, 1])` | places the gate on the listed qubits; a gate is a box until it is decomposed |
| `cx · rz · cx` | the core of MyGate: a ZZ rotation (see the box after the next cell) |

</details>


### Decomposing Custom Gates

In [ ]:
# Decompose custom gate
qc_decomposed = qc_custom.decompose()              # expand every box ONE level (MyGate -> h, cx, rz, cx)

print("Decomposed circuit:")
print(f"  Original depth: {qc_custom.depth()}")
print(f"  Decomposed depth: {qc_decomposed.depth()}")

# the core of MyGate is a two-qubit ZZ rotation: check it with the unitary matrices
from qiskit.quantum_info import Operator           # the unitary matrix of a circuit
core = QuantumCircuit(2); core.cx(0, 1); core.rz(np.pi/4, 1); core.cx(0, 1)
ref = QuantumCircuit(2); ref.rzz(np.pi/4, 0, 1)    # RZZ(θ) = exp(−i θ/2 Z⊗Z)
print(f"  CX·RZ(π/4)·CX equals RZZ(π/4): {Operator(core).equiv(Operator(ref))}")   # equiv: equal up to a global phase
display(draw(qc_decomposed)); print("↑ the boxes expanded: each MyGate became H, CX, RZ(π/4), CX")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc_custom.decompose()` | replaces each composite gate by its definition (one level); `decompose(reps=2)` would go deeper |
| depth 3 → 9 | a box counts as one time step; after expansion the 4 gates inside each MyGate are serial, and the second MyGate must wait for the first on qubit 1: 1 + 4 + 4 = 9 |
| `Operator(core).equiv(Operator(ref))` | compare two circuits by their 4×4 unitaries, ignoring a global phase |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Why CX · RZ · CX is a ZZ rotation.** Conjugating by a CNOT (control 0, target 1) maps $Z_1 \to Z_0 Z_1$: on the basis state $|q_1 q_0\rangle$ the target qubit carries the parity $q_0 \oplus q_1$ in the middle of the sandwich. Hence

$$\mathrm{CX}\;\big(R_Z(\varphi)\otimes I\big)_{q_1}\;\mathrm{CX} = e^{-i\frac{\varphi}{2} Z_0 Z_1} = R_{ZZ}(\varphi),$$

a phase $e^{\mp i\varphi/2}$ for even / odd parity of the two qubits. With $\varphi = \pi/4$: phases $e^{-i\pi/8}$ on $|00\rangle, |11\rangle$ and $e^{+i\pi/8}$ on $|01\rangle, |10\rangle$ — the cell prints `True`. This is the building block of the ZZ feature map and of QAOA cost layers (notebook 03).

</details>


## 🔬 Part 6: Circuit Execution

### Simulating Circuits

In [ ]:
from qiskit.visualization import plot_histogram   # bar chart of measurement counts

# Create example circuit
qc_sim = QuantumCircuit(3, 3)                      # 3 qubits + a classical register 'c' of 3 bits
qc_sim.h(0)
qc_sim.cx(0, 1)
qc_sim.cx(1, 2)                                    # GHZ state (|000⟩ + |111⟩)/√2
qc_sim.measure_all()                               # adds a barrier + a NEW 3-bit register 'meas' and measures into it ('c' stays unused)
display(draw(qc_sim)); print("↑ the GHZ circuit; measure_all() wrote into a new register 'meas' — the original register c is never filled")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(3, 3)` | 3 qubits and a 3-bit classical register named `c` |
| `qc_sim.measure_all()` | barrier + measurement of every qubit into a **new** register `meas`; to use the existing `c`, write `qc_sim.measure([0, 1, 2], [0, 1, 2])` instead |
| `h`, `cx`, `cx` | the 3-qubit GHZ state: only 000 and 111 can come out, each with probability 1/2 |

</details>


In [ ]:
# Run on the IQM backend chosen by TARGET
# run_counts = transpile_to_IQM (native r = PRX and cz gates, routed on the chip) + backend.run + get_counts()
counts_all, transpiled = run_counts(qc_sim, TARGET, shots=1024, return_transpiled=True)
print(f"transpiled for IQM: {dict(transpiled.count_ops())}, depth {transpiled.depth()}")   # 'r' = PRX rotation, 'cz' = controlled-Z
print(counts_all)                      # keys = one group of bits per classical register, 'meas' first, then the empty 'c'

def run_circuit(circuit, shots=1024):
    """Transpile to the IQM native gates + run on TARGET + counts in one call (used in the In-depth analysis)."""
    return run_counts(circuit, TARGET, shots=shots)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(qc_sim, TARGET, shots=1024, return_transpiled=True)` | the helper's pipeline: `transpile_to_IQM` (native `r` = PRX and `cz`, routed on the chip's square lattice) → `backend.run` → counts; the transpiled circuit comes back too |
| `count_ops()`: `r`, `cz` | each CNOT became one CZ between single-qubit rotations; `measure` and `barrier` are kept |
| the keys of `counts_all` | one group per classical register, separated by a space: the 'meas' bits, then the never-written register 'c' (the next cell keeps only 'meas') |
| `TARGET` | `fake_apollo` = local noisy model (a few 001, 110 … appear); `garnet:mock` = random bitstrings; `garnet` / `emerald` = real QPU (IQM Resonance caps circuits at 10 000 native instructions) |

</details>


In [ ]:
counts = {}                                         # keep only the 'meas' register (created by measure_all)
for key, n in counts_all.items():
    bits = key.split()[0]                           # split on spaces; [0] = the leftmost group = the last-added register 'meas'
    counts[bits] = counts.get(bits, 0) + n          # merge keys that differ only in the (empty) register 'c'

print("Simulation results:")
display(plot_histogram(counts)); print(f"↑ GHZ statistics: 000 → {counts.get('000', 0)}, 111 → {counts.get('111', 0)} of 1024 shots, other outcomes → {1024 - counts.get('000', 0) - counts.get('111', 0)}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `key.split()[0]` | Qiskit-style counts keys list the registers last-added first: 'meas' is the leftmost group |
| `counts.get(bits, 0) + n` | accumulate into a dict, starting from 0 for a new key |
| the outcomes other than 000 / 111 | gate and readout errors of the (simulated) device: an ideal GHZ state never gives them |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**What to expect from 1024 shots.** The GHZ circuit prepares $\frac{1}{\sqrt2}(|000\rangle + |111\rangle)$: $H$ makes $q_0$ a superposition and each CNOT copies its value to the next qubit. The Born rule gives $P(000) = P(111) = |1/\sqrt2|^2 = 1/2$ and zero for the other six strings.

Each shot is a coin flip, so the number of 000 outcomes is binomial with mean $1024\cdot\frac12 = 512$ and standard deviation $\sqrt{1024\cdot\frac12\cdot\frac12} = 16$: a result like 497 / 527 is perfectly normal, while 400 / 624 would signal a problem. The caption above prints the actual numbers; on an ideal simulator the "other outcomes" are exactly 0.

</details>


## 📊 Part 7: Circuit Analysis

### Circuit Properties

In [ ]:
# Analyze circuit
qc_analyze = QuantumCircuit(3)
qc_analyze.h(0)
qc_analyze.cx(0, 1)
qc_analyze.cx(1, 2)
qc_analyze.rz(0.5, 2)                              # angle in radians
qc_analyze.cx(1, 2)
qc_analyze.cx(0, 1)                                # the CNOT ladder undone: together a 3-qubit ZZZ rotation
display(draw(qc_analyze)); print("↑ a CNOT ladder, RZ(0.5) on the last qubit, the ladder in reverse: every gate waits for the previous one")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `cx` ladder + `rz` + reversed ladder | the standard way to build exp(−i θ/2 Z⊗Z⊗Z): the ladder collects the parity of the three qubits on q2 |
| `rz(0.5, 2)` | a rotation by 0.5 rad around z on qubit 2 |

</details>


In [ ]:
print("Circuit Analysis:")
print("="*60)
print(f"  Total qubits: {qc_analyze.num_qubits}")
print(f"  Depth: {qc_analyze.depth()}")                              # longest path: all 6 gates are in sequence
print(f"  Size (gates): {qc_analyze.size()}")                        # total number of operations
print(f"  Width: {qc_analyze.width()}")                              # qubits + classical bits
print(f"  Number of nonlocal gates: {qc_analyze.num_nonlocal_gates()}")   # gates on ≥ 2 qubits (the expensive ones on hardware)
print("="*60)

# Count operations
ops = qc_analyze.count_ops()                       # an ordered dict {gate name: count}
print("\nOperation counts:")
for gate, count in ops.items():
    print(f"  {gate}: {count}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `depth()` = 6, `size()` = 6 | no two gates can run in parallel here, so depth = size |
| `width()` | qubits + classical bits = 3 (no classical bits) |
| `num_nonlocal_gates()` = 4 | the two-qubit gates: on real hardware they are ~10× noisier than single-qubit ones, so this is the number to minimise |
| `count_ops()` | a histogram of gate names: {'cx': 4, 'h': 1, 'rz': 1} |

</details>


## 🎓 Practical Examples

### Example 1: GHZ State

In [ ]:
def create_ghz(n_qubits):
    """Create n-qubit GHZ state."""
    qc = QuantumCircuit(n_qubits, name=f'GHZ_{n_qubits}')
    qc.h(0)
    for i in range(n_qubits - 1):                  # range(n-1) = 0, 1, …, n-2
        qc.cx(i, i + 1)                            # a chain of CNOTs copies the value of q0 along the register
    return qc

# Create 4-qubit GHZ
ghz4 = create_ghz(4)
print("4-qubit GHZ state:")
display(draw(ghz4)); print("↑ (|0000⟩ + |1111⟩)/√2: one H and a chain of 3 CNOTs — depth grows linearly with the number of qubits")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `def create_ghz(n_qubits)` | a circuit *factory*: a function that returns a new circuit for any size |
| `for i in range(n_qubits - 1)` | n−1 CNOTs, each one step later than the previous: depth n |

</details>


### Example 2: Quantum Fourier Transform

In [ ]:
def qft(n):
    """Create n-qubit QFT circuit."""
    qc = QuantumCircuit(n, name=f'QFT_{n}')
    
    for j in range(n):
        qc.h(j)
        for k in range(j+1, n):
            qc.cp(np.pi/2**(k-j), k, j)            # controlled phase π/2^(k−j), control k, target j
    
    # Swap qubits
    for i in range(n//2):                          # // = integer division
        qc.swap(i, n-i-1)                          # reverse the qubit order
    
    return qc

# Create 3-qubit QFT
qft3 = qft(3)
print("3-qubit QFT:")
display(draw(qft3)); print(f"↑ H, controlled phases π/2 and π/4, then a swap of q0 and q2: {dict(qft3.count_ops())}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.cp(angle, k, j)` | controlled phase: multiplies \|11⟩ of qubits k, j by e^{i·angle}; it is symmetric in the two qubits |
| `np.pi/2**(k-j)` | `**` is the power operator: π/2, π/4, π/8 … for qubits 1, 2, 3 … apart |
| gate count | n Hadamards + n(n−1)/2 controlled phases + ⌊n/2⌋ swaps: 3 + 3 + 1 for n = 3 |
| ordering | this textbook circuit treats qubit 0 as the *most* significant bit; Qiskit's library QFT uses the opposite convention (In-depth task 2) |

</details>


### Example 3: Variational Circuit

In [ ]:
def variational_circuit(n_qubits, n_layers):
    """Create parameterized variational circuit."""
    n_params = 2 * n_qubits * n_layers             # two angles (RY, RZ) per qubit per layer
    params = ParameterVector('θ', n_params)
    
    qc = QuantumCircuit(n_qubits)
    param_idx = 0
    
    for layer in range(n_layers):
        # Rotation layer
        for qubit in range(n_qubits):
            qc.ry(params[param_idx], qubit)
            param_idx += 1                         # x += 1 is shorthand for x = x + 1
            qc.rz(params[param_idx], qubit)
            param_idx += 1
        
        # Entangling layer
        for qubit in range(n_qubits - 1):
            qc.cx(qubit, qubit + 1)
    
    return qc

# Create variational circuit
var_qc = variational_circuit(3, 2)
print(f"Variational circuit with {var_qc.num_parameters} parameters:")
display(draw(var_qc)); print("↑ two layers of (RY, RZ) rotations on every qubit, each followed by a CNOT chain: 12 free angles")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `n_params = 2 * n_qubits * n_layers` | 2 × 3 × 2 = 12 angles |
| rotation layer + entangling layer | the generic shape of a *hardware-efficient ansatz* (VQE, QML): rotations explore each qubit's sphere, CNOTs create correlations |
| `param_idx` | a running counter that hands out the next unused parameter |

</details>


## 🎓 Exercises

### Exercise 1: Bell States (Easy)
Create all four Bell states (Φ+, Φ-, Ψ+, Ψ-) as separate circuits.

```python
# Your solution here
```

### Exercise 2: Custom Toffoli (Medium)
Build a Toffoli (CCX) gate using only single-qubit gates and CNOT. Verify it works correctly.

### Exercise 3: Quantum Adder (Hard)
Implement a 2-bit quantum adder circuit that adds two 2-bit numbers.

## 🔬 In-depth analysis

(Different from the 🎓 Exercises above: those ask you to build new circuits; these measure and compare the circuits of this notebook.)

1. **GHZ under the microscope.** For $n = 2, \dots, 6$ build `create_ghz(n)`, print its depth and number of CNOTs, add `measure_all()` and run it with `run_circuit(circuit, shots=1024)`. Print the fraction of shots that land on $0\cdots0$ or $1\cdots1$: it is 1 on an ideal simulator — what happens on a noisy device as $n$ grows?
2. **Two QFT conventions.** Compare the unitary of `qft(3)` with the library gate `QFTGate(3)` using `Operator(...).equiv(...)`. They differ — show that they agree once the qubit order of `qft3` is reversed (`qft3.reverse_bits()`), and explain why with the little-endian convention.
3. **Sampling vs exact.** Bind the 12 parameters of `var_qc` to random angles (`np.random.default_rng(7)`), compute the exact outcome probabilities with `Statevector(...).probabilities_dict()`, and compare them with the frequencies of 4096 shots of the measured circuit via the total variation distance $\frac12\sum_s |p(s) - f(s)|$.

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution
from qiskit.circuit.library import QFTGate          # the library QFT (Qiskit ≥ 1.2)
from qiskit.quantum_info import Operator, Statevector

# (1) GHZ states of growing size: cost and measured populations
for n in range(2, 7):
    ghz = create_ghz(n)
    depth, n_cx = ghz.depth(), ghz.count_ops().get("cx", 0)
    ghz.measure_all()                                # in place: adds the 'meas' register
    c = run_circuit(ghz, shots=1024)
    good = c.get("0" * n, 0) + c.get("1" * n, 0)     # "0"*n = '00…0'
    print(f"(1) n = {n}: depth {depth}, {n_cx} CNOTs, P(0…0) + P(1…1) = {good / 1024:.3f}")

# (2) textbook QFT vs library QFT
same = Operator(qft3).equiv(Operator(QFTGate(3)))
same_rev = Operator(qft3.reverse_bits()).equiv(Operator(QFTGate(3)))
print(f"(2) qft(3) == QFTGate(3): {same};  qft(3) with the qubit order reversed == QFTGate(3): {same_rev}")

# (3) exact probabilities vs 4096 samples of a random instance of the ansatz
rng = np.random.default_rng(7)                       # seeded random generator: reproducible angles
values = rng.uniform(0, 2 * np.pi, var_qc.num_parameters)
bound = var_qc.assign_parameters(values)             # a list binds the ParameterVector in order
exact = Statevector(bound).probabilities_dict()
meas = bound.copy(); meas.measure_all()
freq = {k: v / 4096 for k, v in run_circuit(meas, shots=4096).items()}   # dict comprehension: counts -> frequencies
tvd = 0.5 * sum(abs(exact.get(s, 0) - freq.get(s, 0)) for s in set(exact) | set(freq))   # | = union of the two key sets
print(f"(3) total variation distance exact vs sampled: {tvd:.3f}  (shot noise alone gives ~0.01–0.02 for 8 outcomes)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `c.get("0" * n, 0)` | the counts of the all-zero string; string repetition builds the key for any n |
| `Operator(a).equiv(Operator(b))` | compares the 2ⁿ×2ⁿ unitaries up to a global phase |
| `qft3.reverse_bits()` | the same circuit with qubit i relabelled n−1−i: it converts between the textbook (big-endian) and Qiskit (little-endian) QFT |
| `rng.uniform(0, 2*np.pi, 12)` | 12 random angles; with a fixed seed every run (and both editions) uses the same ones |
| total variation distance | half the L1 distance between two distributions: 0 = identical, 1 = disjoint; device noise adds to the shot-noise floor |

</details>


## 📖 References

### Documentation
1. **QuantumCircuit API**: https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit
2. **Circuit Library**: https://docs.quantum.ibm.com/api/qiskit/circuit_library
3. **Circuit Composition**: https://docs.quantum.ibm.com/guides/construct-circuits

---

**Course**: Quantum Computing Laboratory  
**License**: MIT  
**Version**: 2.2.0  
**Last Updated**: November 2025